# Naive Bayes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/EnriqueVilchezL/machine-learning/blob/main/book/03-supervised/naive-bayes/naive-bayes-notebook.ipynb)

Companion notebook for the *Naive Bayes* slides. Each idea is written first by hand
with NumPy and then with the scikit-learn class that does the same job.

**Data:** the Titanic passenger list for counting categories, and the Wine recognition data
for numeric features.

In [1]:
import numpy as np
import seaborn as sns
from scipy.special import softmax
from sklearn import naive_bayes
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

RANDOM_STATE = 42
ALPHA = 1
VAR_SMOOTHING = 1e-9

## 1. Bayes' rule

$P(c \mid \mathbf{x}) = \dfrac{P(\mathbf{x} \mid c)\,P(c)}{P(\mathbf{x})}$. With the numbers from the slides:

In [2]:
prior_spam, likelihood_spam, likelihood_ham = 0.3, 0.8, 0.0857
evidence = likelihood_spam * prior_spam + likelihood_ham * (1 - prior_spam)   # P(word)
posterior_spam = likelihood_spam * prior_spam / evidence
print(f"P(spam | word) = {posterior_spam:.2f}")

P(spam | word) = 0.80


## 2. Counting with smoothing

Naive Bayes multiplies one probability per feature: $\hat{y} = \arg\max_c P(c)\prod_j P(x_j \mid c)$.
Each probability is a count, with Laplace smoothing so no count is zero:
$P(x_j = v \mid c) = \dfrac{N_{jvc} + \alpha}{N_c + \alpha V_j}$, where $V_j$ is the number of values of feature $j$.
`fit` only counts; `predict_proba` adds the logs and rescales so the classes add up to 1.

In [3]:
class CategoricalNB:
    """Naive Bayes for categorical features coded 0, 1, 2, ..., with Laplace smoothing."""

    def __init__(self, alpha: float = 1.0):
        self.alpha = alpha

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.class_log_prior_ = np.log([np.mean(y == c) for c in self.classes_])
        self.feature_log_prob_ = []
        for j in range(X.shape[1]):
            n_values = X[:, j].max() + 1                                  # V_j
            counts = np.zeros((len(self.classes_), n_values))
            for i, c in enumerate(self.classes_):
                counts[i] = np.bincount(X[y == c, j], minlength=n_values)   # N_jvc
            smoothed = counts + self.alpha                                # N_jvc + alpha
            totals = smoothed.sum(axis=1, keepdims=True)                  # N_c + alpha V_j
            self.feature_log_prob_.append(np.log(smoothed / totals))
        return self

    def predict_proba(self, X):
        log_likelihood = np.zeros((len(X), len(self.classes_)))
        for j, log_prob in enumerate(self.feature_log_prob_):
            log_likelihood += log_prob[:, X[:, j]].T                      # + log P(x_j | c)
        scores = self.class_log_prior_ + log_likelihood                   # log P(c) + sum_j ...
        return softmax(scores, axis=1)

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]


titanic = sns.load_dataset("titanic")[["sex", "pclass", "embarked", "survived"]].dropna()
X = OrdinalEncoder().fit_transform(titanic[["sex", "pclass", "embarked"]]).astype(int)
y = titanic["survived"].to_numpy()

by_hand = CategoricalNB(alpha=ALPHA).fit(X, y)

In [4]:
model = naive_bayes.CategoricalNB(alpha=ALPHA).fit(X, y)
print("same probabilities:", np.allclose(by_hand.predict_proba(X), model.predict_proba(X)))

same probabilities: True


Adding logs instead of multiplying probabilities gives the same winner and avoids numbers that are too small for the computer.

## 3. Numeric features: the bell curve

For a number, $P(x_j \mid c)$ is a normal curve with the class mean $\mu_{jc}$ and variance $\sigma_{jc}^2$.
scikit-learn adds a tiny amount to every variance, a share `VAR_SMOOTHING` of the largest feature variance, so none is zero.

In [5]:
class GaussianNB:
    """Naive Bayes with one bell curve per feature and class."""

    def __init__(self, var_smoothing: float = 1e-9):
        self.var_smoothing = var_smoothing

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.class_prior_ = np.array([np.mean(y == c) for c in self.classes_])
        self.theta_ = np.array([X[y == c].mean(axis=0) for c in self.classes_])
        extra_variance = self.var_smoothing * X.var(axis=0).max()
        self.var_ = np.array([X[y == c].var(axis=0) for c in self.classes_]) + extra_variance
        return self

    def predict_proba(self, X):
        log_prior = np.log(self.class_prior_)                             # log P(c)
        squared_gap = (X[:, None, :] - self.theta_) ** 2                  # (x_j - mu_jc)^2
        log_density = -0.5 * (np.log(2 * np.pi * self.var_) + squared_gap / self.var_)
        log_likelihood = log_density.sum(axis=2)                          # sum_j log P(x_j | c)
        scores = log_prior + log_likelihood
        return softmax(scores, axis=1)

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]

    def score(self, X, y):
        return np.mean(self.predict(X) == y)


wine = load_wine()
X_train, X_test, y_train, y_test = train_test_split(wine.data, wine.target, test_size=0.3,
                                                    random_state=RANDOM_STATE, stratify=wine.target)
by_hand = GaussianNB(var_smoothing=VAR_SMOOTHING).fit(X_train, y_train)

In [6]:
model = naive_bayes.GaussianNB(var_smoothing=VAR_SMOOTHING).fit(X_train, y_train)
same_means = np.allclose(by_hand.theta_, model.theta_)
same_variances = np.allclose(by_hand.var_, model.var_)
print("same means and variances:", same_means and same_variances)
print("same probabilities:", np.allclose(by_hand.predict_proba(X_test), model.predict_proba(X_test)))
accuracy_by_hand = by_hand.score(X_test, y_test)
accuracy_library = model.score(X_test, y_test)
print(f"test accuracy: by hand {accuracy_by_hand:.3f}, scikit-learn {accuracy_library:.3f}")

same means and variances: True
same probabilities: True
test accuracy: by hand 1.000, scikit-learn 1.000


## Exercises

1. Set `ALPHA = 0` in section 2. Which probabilities become 0, and why is that a problem?
2. Add `alone` as a fourth Titanic feature and check the result against `naive_bayes.CategoricalNB` again.
3. Print `by_hand.theta_` for `alcohol` (the first feature) in the Wine data. Which class does a wine with 14% alcohol favor?